# Candidate Generation для поиска услуг Авито

Итоговое решение тестового задания.

**Benchmark Recall@50: `0.818206`.**

Основная идея: не пытаться найти все релевантные объявления одним retriever. Вместо этого объединяются несколько complementary candidate channels, после чего LightGBM выбирает финальные 50 кандидатов.

```text
geo-aware BM25 ───────┐
global semantic ──────┼──> union (~1500) ──> LightGBM ──> top-50
exact geo-semantic ───┤
transition semantic ──┘
```

## 1. Данные и метрика

Используются:

- `train.parquet` — положительные query–item пары;
- `benchmark_queries.parquet` — 2 452 запроса;
- `benchmark_items.parquet` — 189 212 объявлений.

Метрика — macro **Recall@50**:

\[
Recall@50 = \frac{1}{N}\sum_q \frac{|Top50_q \cap Relevant_q|}{|Relevant_q|}
\]

Для candidate generation важнее всего не потерять релевантное объявление до финального ranking.

## 2. Валидация

Random split строк использовать нельзя: одинаковые тексты запросов часто повторяются в train.

Поэтому split выполнялся по нормализованному `search_query`:

- lowercase;
- `ё → е`;
- нормализация пробелов.

Validation-тексты полностью исключались из fit. Для model selection использовался fixed primary из 3 000 новых текстов, а финальный reranker дополнительно проверялся на fresh confirmation из ещё 3 000 текстов.

## 3. BM25 baseline

Начальный retriever — полевой BM25:

| Поле | Вес |
|---|---:|
| title | 4.0 |
| params | 1.0 |
| description | 0.4 |

`k1=1.2`, `b=0.75`.

**Primary Recall@50 = `0.320390`.**

## 4. География

Error analysis показал сильный geographic signal. При этом hard filter по location невозможен: часть релевантных query–item пар связывает разные location IDs.

Используются мягкие признаки:

- exact location: итоговый multiplier `×3`;
- сглаженные train-derived `search_location → item_location` transitions;
- transition strength `0.5`;
- category bonus.

Главное изменение: geography применяется **до top-k**, а не после candidate cutoff.

Результат:

- Recall@50 = `0.536322`;
- Recall@500 = `0.702944`.

## 5. Semantic retrieval

Для перефразирований и слабого lexical overlap используется `intfloat/multilingual-e5-small`.

Query representation:

```text
query: <search_query>. <filters[:200]>
```

Document representation:

```text
passage: <title>. <params[:600]>
```

Retrieval channels:

- global semantic top-1000;
- exact-location semantic top-200;
- transition-location semantic top-100.

Global channel сохраняется всегда: география не превращается в hard filter.

## 6. Candidate pool

Финальный union:

```text
geo-BM25-500
∪ global-semantic-1000
∪ exact-location-semantic-200
∪ transition-location-semantic-100
```

| Срез | Candidate recall |
|---|---:|
| Primary | `0.915611` |
| Fresh confirmation | `0.925850` |

После этого основным bottleneck стал уже ranking внутри найденного пула.

## 7. LightGBM reranker

`LGBMRanker` с `lambdarank` использует 39 числовых признаков:

- retrieval scores и ranks;
- semantic cosine;
- exact/transition geography;
- channel presence/count;
- lexical overlap query с title/params/description;
- длины;
- rating/reviews/price/contact flags.

ID в качестве признаков не используются.

## 8. Hard-negative mining

V1 уже заметно улучшил Recall@50, но при inference видел около 1500 кандидатов на запрос.

Для V2 frozen V1 ранжирует полный training pool. Затем на каждый query сохраняются:

- все positives;
- 200 наиболее высоко оценённых false candidates;
- RRF-дополнение сильными retrieval-кандидатами до 250 строк.

Набор признаков и модель остаются теми же — меняется только сложность negatives.

## 9. Результаты

| Метод | Primary Recall@50 | Fresh Recall@50 |
|---|---:|---:|
| BM25 baseline | `0.320390` | — |
| Geography-aware BM25 | `0.536322` | `0.534521` |
| Reranker v1 | `0.773538` | `0.788533` |
| **Reranker v2** | **`0.788886`** | **`0.813367`** |

Benchmark:

| Submission | Recall@50 |
|---|---:|
| geography-aware lexical | `0.519964` |
| **hybrid + reranker v2** | **`0.818206`** |

Fresh validation и hidden benchmark отличаются менее чем на 0.5 п.п.

## 10. Воспроизведение

Вся финальная логика собрана в `solution.py`. На первом запуске скрипт:

1. воспроизводит grouped split;
2. строит local BM25 corpus/index;
3. считает E5 embeddings;
4. формирует ranker train/internal query groups;
5. обучает v1;
6. выполняет hard-negative mining и обучает v2;
7. строит benchmark retrieval channels;
8. ранжирует union и записывает `answer.csv`.

Этапы кэшируются в `artifacts/repro`.

In [ ]:
from pathlib import Path
from solution import run_pipeline

# Полный end-to-end запуск. Первый прогон дорогой, повторный использует кэши.
result = run_pipeline(
    data_dir=Path("data"),
    work_dir=Path("artifacts/repro"),
    output_path=Path("answer.csv"),
    model_path=None,  # можно указать локальную директорию multilingual-e5-small
)

result

## 11. Итог

Финальное качество получилось не из одного «магического» алгоритма, а из последовательного устранения bottleneck'ов:

1. BM25 дал интерпретируемый lexical baseline;
2. раннее применение geography сильно подняло coverage;
3. global semantic retrieval вернул перефразирования;
4. geo-conditioned semantic retrieval поднял candidate recall выше 0.91;
5. supervised reranking научился выбирать top-50;
6. hard-negative mining улучшил обучение на самых опасных ошибках V1.

**Итоговый benchmark Recall@50: `0.818206`.**